In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd
import os
import locale
from datetime import datetime, timedelta
import unicodedata

In [ ]:
# parse the dates

MONTHS_FR = {
    "janvier": "01",
    "février": "02", "fevrier": "02",
    "mars": "03",
    "avril": "04",
    "mai": "05",
    "juin": "06",
    "juillet": "07",
    "août": "08", "aout": "08",
    "septembre": "09",
    "octobre": "10",
    "novembre": "11",
    "décembre": "12", "decembre": "12"
}

def parse_french_date_with_month_map(text):

    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn').lower()

    for month_txt, month_num in MONTHS_FR.items():
        if month_txt in text:
            text = text.replace(month_txt, month_num)
            break

    try:
        return datetime.strptime(text.strip(), "%d %m %Y %H:%M")
    except ValueError:
        return None

In [ ]:
user_agent = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/115.0.0.0 Safari/537.36"
)

options = Options()
options.add_argument(f"user-agent={user_agent}")

In [ ]:
# Selenium Manager automatically handles ChromeDriver
driver = webdriver.Chrome(options=options)

driver.get("LINK") #add link

time.sleep(3)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

In [ ]:
# number of pages to scrape

try:
    pagination_links = driver.find_elements (By.CSS_SELECTOR, "ul.pagination a.page-link")

    page_numbers= []
    for page in pagination_links:
        txt = page.text.strip()
        if txt.isdigit():
            page_numbers.append(int(txt))
    
    if page_numbers: 
        total_pages = max(page_numbers)
        
except NoSuchElementException:
    total_pages = 1

print("total number of pages : ",total_pages)

In [ ]:

import time
from datetime import datetime, timedelta
from selenium.webdriver.common.by import By

# Scraping
tender_list = []

# No need for locale.setlocale:
# parse_french_date_with_month_map() handles the French month names.

for i in range(1, total_pages + 1):
    driver.get(f"https://www.arpce.dz/fr/avis?page={i}")
    time.sleep(3)

    tenders = driver.find_elements(By.CSS_SELECTOR, "li.list-group-item")

    for tender in tenders:
        try:
            # Publication date
            date = tender.find_element(
                By.CSS_SELECTOR, "span.card-date"
            ).text.strip()

            pub_date = parse_french_date_with_month_map(date)

            # Skip tenders with no date or older than 30 days
            if pub_date is None:
                continue

            if datetime.now() - pub_date > timedelta(days=30):
                continue

            # Title without the text coming from any child elements
            h5 = tender.find_element(
                By.CSS_SELECTOR, "h5.card-title"
            )

            title = driver.execute_script("""
                const el = arguments[0];
                let text = "";
                for (const node of el.childNodes) {
                    if (node.nodeType === Node.TEXT_NODE) {
                        text += node.textContent;
                    }
                }
                return text.trim();
            """, h5)

            # Description
            p = tender.find_element(
                By.CSS_SELECTOR, "p.card-text"
            ).text.strip()

            full_title = title + "\n" + p

            # Tender link
            link = tender.find_element(
                By.CSS_SELECTOR, "a.btn-read"
            ).get_attribute("href")

            # Save the tender
            tender_record = {
                "title": full_title,
                "publication_date": pub_date.strftime("%d/%m/%Y"),
                "expiration_date": None,
                "link": link,
                "country": "Algeria"
            }

            tender_list.append(tender_record)

        except Exception as e:
            print(f"Error while processing a tender: {e}")
            continue

print(f"Total number of tenders collected: {len(tender_list)}")
print(tender_list)

In [ ]:
driver.quit()

In [ ]:
# save file

df = pd.DataFrame(tender_list)
df.to_excel("../2_scraped_files/Algeria.xlsx", index=False) # modify